# Coffee17 preprocessing — one-time OOF
Run only after validation decision authorizes it. Inference only.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import importlib, os, shutil, subprocess, sys, urllib.request
from pathlib import Path
BRANCH='codex/preprocessing-study-v1'; REPO=Path('/content/coffee-bean-classification'); WORK=Path('/content')
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-classification.git',str(REPO)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(REPO/'requirements/preprocessing-study.txt')],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','-e',str(REPO)],check=True)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
from bilinear_lmmd.core.drive_project import resolve_drive_project_root
from bilinear_lmmd.data.preparation.prepare_coffee17 import DATASET_URL
from bilinear_lmmd.data.preparation.audit_coffee17_provenance import audit_coffee17_provenance
from bilinear_lmmd.experiments.run_preprocessing_oof import run_oof
PROJECT=resolve_drive_project_root(); DATA=PROJECT/'evidence/coffee17-preprocessing-data-v1'; AUTH=PROJECT/'evidence/coffee17-preprocessing-primary-v1/preprocessing_primary_confirmation.json'; EXPERIMENTS=PROJECT/'experiments/coffee17-preprocessing-primary-v1'; OOF=PROJECT/'oof/coffee17-preprocessing-primary-v1'
ARCHIVE=WORK/'coffee17_original.zip'
if not ARCHIVE.is_file():
    req=urllib.request.Request(DATASET_URL,headers={'User-Agent':'Mozilla/5.0'})
    with urllib.request.urlopen(req) as response, ARCHIVE.open('wb') as output: shutil.copyfileobj(response,output)
CANONICAL=WORK/'coffee17_original_v1'; PROV=WORK/'coffee17_oof_provenance'
if CANONICAL.exists(): shutil.rmtree(CANONICAL)
if PROV.exists(): shutil.rmtree(PROV)
audit_coffee17_provenance(ARCHIVE,PROV,canonical_root=CANONICAL)
result=run_oof(canonical_root=CANONICAL,clean_manifest=DATA/'clean_manifest.json',fold_manifest=DATA/'fold_manifest.json',authority_path=AUTH,experiments_root=EXPERIMENTS,output_root=OOF,authorize_test=True)
print('OOF COMPLETE:',result)
